### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [58]:
%pip install numpy scikit-learn

/home/qufa6/proyectos/nlp/nlp-fq/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [59]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [60]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [61]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [62]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [63]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [64]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [65]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [66]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [67]:
#tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [68]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [69]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [70]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [71]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [72]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [73]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [74]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [75]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [76]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [77]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [78]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [79]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [80]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


In [81]:
# TÚ CODIGO AQUÍ

### Resolucion Item 1:

En esta sección vamos a vectorizar 5 documentos seleccionados al azar del dataset de entrenamiento, y luego mediremos su similaridad con todos los demás documentos. Analizaremos los 5 documentos más similares para cada uno y verificaremos si la similaridad tiene sentido de acuerdo al contenido del texto y su etiqueta de clasificación.

In [82]:
# Seleccionar 5 documentos al azar
np.random.seed(42)  # Para reproducibilidad
documentos_indices = np.random.choice(len(newsgroups_train.data), 5, replace=False)
print(f"Índices de documentos seleccionados: {documentos_indices}")

Índices de documentos seleccionados: [7492 3546 5582 4793 3813]


In [83]:
# Para cada documento seleccionado, calcular similaridad y encontrar los 5 más similares
resultados_similaridad = {}

for idx_doc in documentos_indices:
    # Calcular similaridad coseno con todos los documentos
    cossim = cosine_similarity(X_train[idx_doc], X_train)[0]
    
    # Obtener índices de los 5 documentos más similares (excluyendo el mismo documento)
    indices_ordenados = np.argsort(cossim)[::-1]
    top_5_similares = indices_ordenados[1:6]  # [1:6] para excluir el documento mismo (index 0)
    
    resultados_similaridad[idx_doc] = {
        'similitudes': cossim[top_5_similares],
        'indices': top_5_similares
    }

#### Análisis de cada documento:

In [84]:
# Analizar cada documento seleccionado
for idx_doc in documentos_indices:
    print(f"\n{'='*80}")
    print(f"DOCUMENTO {idx_doc}")
    print(f"{'='*80}")
    
    # Mostrar el documento original
    print(f"\nClase: {newsgroups_train.target_names[y_train[idx_doc]]}")
    print(f"\nContenido del documento:")
    print(newsgroups_train.data[idx_doc][:300] + "...")  # Primeros 300 caracteres
    
    # Mostrar los 5 más similares
    print(f"\n\n5 Documentos más similares:")
    print("-" * 80)
    
    top_5 = resultados_similaridad[idx_doc]
    
    for rank, (sim_idx, similitud) in enumerate(zip(top_5['indices'], top_5['similitudes']), 1):
        clase = newsgroups_train.target_names[y_train[sim_idx]]
        print(f"\n{rank}. Similaridad: {similitud:.4f}")
        print(f"   Clase: {clase}")
        print(f"   Contenido: {newsgroups_train.data[sim_idx][:250]}...")


DOCUMENTO 7492

Clase: comp.sys.mac.hardware

Contenido del documento:
Could someone please post any info on these systems.

Thanks.
BoB
-- 
---------------------------------------------------------------------- 
Robert Novitskey | "Pursuing women is similar to banging one's head
rrn@po.cwru.edu  |  against a wall...with less opportunity for reward" ...


5 Documentos más similares:
--------------------------------------------------------------------------------

1. Similaridad: 0.6665
   Clase: comp.sys.mac.hardware
   Contenido: Hey everybody:

   I want to buy a mac and I want to get a good price...who doesn't?  So,
could anyone out there who has found a really good deal on a Centris 650
send me the price.  I don't want to know where, unless it is mail order or
areound clev...

2. Similaridad: 0.3476
   Clase: comp.sys.ibm.pc.hardware
   Contenido: Hay all:

    Has anyone out there heard of any performance stats on the fabled p24t.
 I was wondering what it's performance compared t

#### Conclusión y análisis de resultados:

**Resultados observados:**

La vectorización TF-IDF captura relativamente bien las relaciones semánticas entre documentos, pero con resultados variables según el dominio:

- **Éxito en categorías temáticas claras:** Los documentos de `comp.sys.mac.hardware` (7492) y `misc.forsale` (5582) muestran alta similaridad con documentos de su misma clase. El primer caso presenta una similaridad de 0.67 con otro documento de hardware Mac, confirmando que documentos sobre el mismo tema se agrupan efectivamente.

- **Confusión en temas técnicos relacionados:** El documento 3546 (`comp.os.ms-windows.misc`) tiene sus 5 documentos más similares clasificados como `comp.sys.ibm.pc.hardware`. Esto indica que comparten vocabulario técnico (DMA, controladores, etc.) aunque pertenecen a categorías distintas. La vectorización captura la similitud léxica pero no la diferenciación temática fina.

- **Baja performance en temas generales/políticos:** Los documentos 4793 (`talk.politics.guns`) y 3813 (`rec.sport.hockey`) muestran similaridades bajas (0.23 y 0.25) con sus vecinos más cercanos. El documento de hockey encuentra sus similares en categorías no relacionadas (religión, ateísmo), sugiriendo que el contenido es muy específico o que la vectorización TF-IDF no captura bien estos temas.

**Conclusión general:** La similaridad coseno con TF-IDF es efectiva para documentos con vocabulario técnico específico y repetitivo, pero menos precisa para temas más generales o cuando los documentos comparten vocabulario por razones externas al tema principal.

### Resolucion item 2: Construir un modelo de clasificación por prototipos (tipo zero-shot)

En este item implementamos un clasificador basado en **prototipos** o **nearest neighbor**. La idea es simple: para cada documento del conjunto de test, encontramos el documento más similar en el conjunto de entrenamiento usando similaridad coseno, y le asignamos la clase de ese documento vecino. 

Esto es un enfoque "zero-shot" porque **no entrenamos un modelo con parámetros**, solo comparamos vectores directamente. Es un método de referencia (baseline) muy útil para comparar con modelos más complejos como Naive Bayes.

**Ventajas:** Interpretable, no requiere entrenamiento, funciona inmediatamente  
**Desventajas:** Lento con conjuntos grandes, sensible al ruido en los datos

In [86]:
# Usar el dataset AG News
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# 1. Cargar dataset
print("Cargando dataset AG News...")
dataset = load_dataset("ag_news")

train_data = dataset["train"]
test_data = dataset["test"]

print(f"Dataset cargado exitosamente")
print(f"Clases: {dataset['train'].features['label'].names}")

Cargando dataset AG News...
Dataset cargado exitosamente
Clases: ['World', 'Sports', 'Business', 'Sci/Tech']


In [87]:
# 2. Crear subset para evaluar rápidamente
N_TRAIN = 3000
N_TEST = 500

X_train_text = train_data["text"][:N_TRAIN]
y_train = np.array(train_data["label"][:N_TRAIN])

X_test_text = test_data["text"][:N_TEST]
y_test = np.array(test_data["label"][:N_TEST])

print(f"Tamaño del conjunto de entrenamiento: {len(X_train_text)} documentos")
print(f"Tamaño del conjunto de test: {len(X_test_text)} documentos")
print(f"Número de clases: {len(dataset['train'].features['label'].names)}")
print(f"\nClases: {dataset['train'].features['label'].names}")

Tamaño del conjunto de entrenamiento: 3000 documentos
Tamaño del conjunto de test: 500 documentos
Número de clases: 4

Clases: ['World', 'Sports', 'Business', 'Sci/Tech']


In [88]:
# 3. Vectorizar usando TF-IDF
print("\nVectorizando documentos...")
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    lowercase=True,
    min_df=2  # Ignorar palabras que aparecen en menos de 2 documentos
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print(f"✓ Vectorización completada")
print(f"  Tamaño vocabulario: {X_train.shape[1]} palabras")
print(f"  Matriz de entrenamiento: {X_train.shape}")
print(f"  Matriz de test: {X_test.shape}")


Vectorizando documentos...
✓ Vectorización completada
  Tamaño vocabulario: 5000 palabras
  Matriz de entrenamiento: (3000, 5000)
  Matriz de test: (500, 5000)


#### Clasificación por prototipos: 1-NN con similaridad coseno

In [89]:
# 4. Clasificación por prototipos (1-NN)
print("\nCalculando similaridad entre documentos de test y entrenamiento...")
print("(Esto puede tomar un momento...)")

# Calcular similaridad coseno: matriz de (n_test, n_train)
similarities = cosine_similarity(X_test, X_train)
print(f"Matriz de similaridad creada: {similarities.shape}")

# Para cada documento de test, encontramos el índice del más similar en train
nearest_indices = np.argmax(similarities, axis=1)

# Asignamos la clase del vecino más cercano
y_pred = y_train[nearest_indices]

print("✓ Predicciones completadas")


Calculando similaridad entre documentos de test y entrenamiento...
(Esto puede tomar un momento...)
Matriz de similaridad creada: (500, 3000)
✓ Predicciones completadas


#### Evaluación del modelo de prototipos:

In [90]:
# 5. Evaluación
from sklearn.metrics import confusion_matrix

accuracy = np.mean(y_pred == y_test)
f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

class_names = dataset['train'].features['label'].names

print("\n" + "="*70)
print("MÉTRICAS DE DESEMPEÑO - CLASIFICADOR POR PROTOTIPOS (1-NN)")
print("="*70)
print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score (macro): {f1_macro:.4f}")
print(f"F1-Score (weighted): {f1_weighted:.4f}")
print("="*70)

# Reporte detallado por clase
print("\nReporte por clase:")
print(classification_report(y_test, y_pred, 
                          target_names=class_names,
                          zero_division=0))


MÉTRICAS DE DESEMPEÑO - CLASIFICADOR POR PROTOTIPOS (1-NN)
Accuracy: 0.7820
F1-Score (macro): 0.7771
F1-Score (weighted): 0.7821

Reporte por clase:
              precision    recall  f1-score   support

       World       0.78      0.83      0.80       122
      Sports       0.86      0.80      0.83       145
    Business       0.70      0.69      0.70       106
    Sci/Tech       0.77      0.80      0.78       127

    accuracy                           0.78       500
   macro avg       0.78      0.78      0.78       500
weighted avg       0.78      0.78      0.78       500



#### Análisis de predicciones correctas e incorrectas:

In [95]:
# Análisis detallado
correctas = y_pred == y_test
incorrectas = ~correctas

# Extraer similaridad del vecino más cercano para cada predicción (sin usar len(X_test))
similitudes_vecino = similarities[np.arange(X_test.shape[0]), nearest_indices]

print(f"Total predicciones: {y_test.shape[0]}")
print(f"Predicciones correctas: {np.sum(correctas)} ({np.sum(correctas)/y_test.shape[0]*100:.1f}%)")
print(f"Predicciones incorrectas: {np.sum(incorrectas)} ({np.sum(incorrectas)/y_test.shape[0]*100:.1f}%)")

print("\nSimilaridad promedio del vecino más cercano:")
print(f"  - Predicciones CORRECTAS: {similitudes_vecino[correctas].mean():.4f}")
print(f"  - Predicciones INCORRECTAS: {similitudes_vecino[incorrectas].mean():.4f}")

print("\nDistribución de similaridades del vecino más cercano:")
print(f"  - Mínimo: {similitudes_vecino.min():.4f}")
print(f"  - Q1: {np.percentile(similitudes_vecino, 25):.4f}")
print(f"  - Mediana: {np.median(similitudes_vecino):.4f}")
print(f"  - Q3: {np.percentile(similitudes_vecino, 75):.4f}")
print(f"  - Máximo: {similitudes_vecino.max():.4f}")

Total predicciones: 500
Predicciones correctas: 391 (78.2%)
Predicciones incorrectas: 109 (21.8%)

Similaridad promedio del vecino más cercano:
  - Predicciones CORRECTAS: 0.4275
  - Predicciones INCORRECTAS: 0.3051

Distribución de similaridades del vecino más cercano:
  - Mínimo: 0.1317
  - Q1: 0.2485
  - Mediana: 0.3471
  - Q3: 0.4937
  - Máximo: 1.0000


In [96]:
# Mostrar 3 predicciones correctas
print("\n" + "="*80)
print("EJEMPLOS DE PREDICCIONES CORRECTAS:")
print("="*80)

idx_correctas = np.where(correctas)[0][:3]
for i, test_idx in enumerate(idx_correctas, 1):
    vecino_idx = nearest_indices[test_idx]
    sim = similitudes_vecino[test_idx]
    clase_real = class_names[y_test[test_idx]]
    clase_pred = class_names[y_pred[test_idx]]
    
    print(f"\n{i}. Predicción CORRECTA (similaridad: {sim:.4f})")
    print(f"   Clase real: {clase_real}")
    print(f"   Clase predicha: {clase_pred}")
    print(f"   Documento test: {X_test_text[test_idx][:100]}...")
    print(f"   Vecino similar: {X_train_text[vecino_idx][:100]}...")

# Mostrar 3 predicciones incorrectas
print("\n" + "="*80)
print("EJEMPLOS DE PREDICCIONES INCORRECTAS:")
print("="*80)

idx_incorrectas = np.where(incorrectas)[0][:3]
for i, test_idx in enumerate(idx_incorrectas, 1):
    vecino_idx = nearest_indices[test_idx]
    sim = similitudes_vecino[test_idx]
    clase_real = class_names[y_test[test_idx]]
    clase_pred = class_names[y_pred[test_idx]]
    
    print(f"\n{i}. Predicción INCORRECTA (similaridad: {sim:.4f})")
    print(f"   Clase real: {clase_real}")
    print(f"   Clase predicha: {clase_pred}")
    print(f"   Documento test: {X_test_text[test_idx][:100]}...")
    print(f"   Vecino similar (incorrecto): {X_train_text[vecino_idx][:100]}...")


EJEMPLOS DE PREDICCIONES CORRECTAS:

1. Predicción CORRECTA (similaridad: 0.5289)
   Clase real: Sci/Tech
   Clase predicha: Sci/Tech
   Documento test: The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - T...
   Vecino similar: The Next Great Space Race: SpaceShipOne and Wild Fire to Go For the Gold (SPACE.com) SPACE.com - A p...

2. Predicción CORRECTA (similaridad: 0.1872)
   Clase real: Sci/Tech
   Clase predicha: Sci/Tech
   Documento test: Calif. Aims to Limit Farm-Related Smog (AP) AP - Southern California's smog-fighting agency went aft...
   Vecino similar: Study: Global Warming Could Change Calif. (AP) AP - Global warming could cause dramatically hotter s...

3. Predicción CORRECTA (similaridad: 0.3094)
   Clase real: Sci/Tech
   Clase predicha: Sci/Tech
   Documento test: Open Letter Against British Copyright Indoctrination in Schools The British Department for Education...
   Vecino similar: RealNetworks Slashes Music-Download

#### Interpretación de ejemplos correctos e incorrectos

Los ejemplos muestran que el clasificador por prototipos funciona bien cuando el documento de test encuentra un vecino de entrenamiento con contenido claramente alineado en tema y vocabulario (por ejemplo, noticias de espacio/ciencia en la clase Sci/Tech). En esos casos, incluso con distinta redacción, la similaridad coseno logra capturar el contexto principal.

En las predicciones incorrectas se observa una similaridad más baja o intermedia, y mezcla entre clases con vocabulario periodístico parecido (por ejemplo, Business vs World o Sci/Tech vs Sports). Esto sugiere que el método 1-NN es sensible a ruido léxico: si el vecino más cercano comparte palabras frecuentes pero no el tema central, la clase se asigna mal.

En conclusión, el enfoque por prototipos es un buen baseline por su simplicidad e interpretabilidad, pero su desempeño depende mucho de la calidad del vecino más cercano. Para mejorar, se podría usar k-NN con k>1 o comparar contra modelos entrenados como Naive Bayes.

### Resolucion Item 3: Naive Bayes para maximizar F1 Macro

En este item se comparan dos modelos de Naive Bayes (Multinomial y Complement) buscando maximizar F1-Score Macro en test.

Para eso se prueban distintas configuraciones del vectorizador TF-IDF y de los modelos.
Importante: no se modifica ngram_range (se mantiene en el valor por defecto unigrama).

In [98]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score, classification_report
from datasets import load_dataset
import numpy as np

# Reutilizamos AG News del Item 2; si no existe en memoria, lo cargamos
if "dataset" not in globals():
    dataset = load_dataset("ag_news")

if "X_train_text" not in globals() or "X_test_text" not in globals():
    N_TRAIN = 3000
    N_TEST = 500
    train_data = dataset["train"]
    test_data = dataset["test"]
    X_train_text = train_data["text"][:N_TRAIN]
    y_train = np.array(train_data["label"][:N_TRAIN])
    X_test_text = test_data["text"][:N_TEST]
    y_test = np.array(test_data["label"][:N_TEST])

class_names = dataset["train"].features["label"].names

print("Datos listos")
print("Train:", len(X_train_text), "Test:", len(X_test_text))
print("Clases:", class_names)

Datos listos
Train: 3000 Test: 500
Clases: ['World', 'Sports', 'Business', 'Sci/Tech']


#### Búsqueda de configuraciones (vectorizador + modelo)
Se realiza una búsqueda manual de hiperparámetros.
En cada experimento:
1. Se vectoriza con TF-IDF
2. Se entrena MultinomialNB o ComplementNB
3. Se evalúa F1 Macro en test

In [99]:
# Configuraciones del vectorizador (sin tocar ngram_range)
vectorizer_grid = [
{"stop_words": "english", "max_features": 5000, "min_df": 1, "max_df": 1.0, "sublinear_tf": False},
{"stop_words": "english", "max_features": 5000, "min_df": 2, "max_df": 1.0, "sublinear_tf": False},
{"stop_words": "english", "max_features": 8000, "min_df": 2, "max_df": 0.95, "sublinear_tf": False},
{"stop_words": "english", "max_features": 8000, "min_df": 2, "max_df": 0.95, "sublinear_tf": True},
{"stop_words": "english", "max_features": 10000, "min_df": 3, "max_df": 0.9, "sublinear_tf": True},
]

# Configuraciones de modelos
model_grid = [
    ("MultinomialNB", MultinomialNB, [{"alpha": 0.1}, {"alpha": 0.5}, {"alpha": 1.0}]),
    ("ComplementNB", ComplementNB, [{"alpha": 0.1}, {"alpha": 0.5}, {"alpha": 1.0}]),
]

resultados = []

exp_id = 0
for vcfg in vectorizer_grid:
    vectorizer_nb = TfidfVectorizer(
        stop_words=vcfg["stop_words"],
        max_features=vcfg["max_features"],
        min_df=vcfg["min_df"],
        max_df=vcfg["max_df"],
        sublinear_tf=vcfg["sublinear_tf"]
        # ngram_range NO se toca
    )

    # Verificación explícita de la restricción
    assert vectorizer_nb.ngram_range == (1, 1)

    Xtr = vectorizer_nb.fit_transform(X_train_text)
    Xte = vectorizer_nb.transform(X_test_text)

    for model_name, model_cls, mcfg_list in model_grid:
        for mcfg in mcfg_list:
            exp_id += 1
            model = model_cls(**mcfg)
            model.fit(Xtr, y_train)
            pred = model.predict(Xte)
            f1m = f1_score(y_test, pred, average="macro")

            resultados.append({
                "exp_id": exp_id,
                "modelo": model_name,
                "alpha": mcfg["alpha"],
                "max_features": vcfg["max_features"],
                "min_df": vcfg["min_df"],
                "max_df": vcfg["max_df"],
                "sublinear_tf": vcfg["sublinear_tf"],
                "f1_macro": f1m,
                "vectorizer": vectorizer_nb,
                "model": model,
                "pred": pred
            })

resultados = sorted(resultados, key=lambda x: x["f1_macro"], reverse=True)

print("Top 10 configuraciones por F1 Macro")
for r in resultados[:10]:
    print(
        f"Exp {r['exp_id']:02d} | {r['modelo']:13s} | alpha={r['alpha']:>3} | "
        f"max_feat={r['max_features']:>5} | min_df={r['min_df']} | max_df={r['max_df']} | "
        f"sublinear_tf={r['sublinear_tf']} | F1_macro={r['f1_macro']:.4f}"
    )

Top 10 configuraciones por F1 Macro
Exp 26 | MultinomialNB | alpha=0.5 | max_feat=10000 | min_df=3 | max_df=0.9 | sublinear_tf=True | F1_macro=0.8621
Exp 02 | MultinomialNB | alpha=0.5 | max_feat= 5000 | min_df=1 | max_df=1.0 | sublinear_tf=False | F1_macro=0.8618
Exp 20 | MultinomialNB | alpha=0.5 | max_feat= 8000 | min_df=2 | max_df=0.95 | sublinear_tf=True | F1_macro=0.8609
Exp 24 | ComplementNB  | alpha=1.0 | max_feat= 8000 | min_df=2 | max_df=0.95 | sublinear_tf=True | F1_macro=0.8601
Exp 08 | MultinomialNB | alpha=0.5 | max_feat= 5000 | min_df=2 | max_df=1.0 | sublinear_tf=False | F1_macro=0.8561
Exp 17 | ComplementNB  | alpha=0.5 | max_feat= 8000 | min_df=2 | max_df=0.95 | sublinear_tf=False | F1_macro=0.8561
Exp 05 | ComplementNB  | alpha=0.5 | max_feat= 5000 | min_df=1 | max_df=1.0 | sublinear_tf=False | F1_macro=0.8556
Exp 06 | ComplementNB  | alpha=1.0 | max_feat= 5000 | min_df=1 | max_df=1.0 | sublinear_tf=False | F1_macro=0.8552
Exp 29 | ComplementNB  | alpha=0.5 | max_fea

#### Mejor configuración encontrada y evaluación final

In [101]:
best = resultados[0]

print("Mejor experimento encontrado")
print("Modelo:", best["modelo"])
print("Alpha:", best["alpha"])
print("max_features:", best["max_features"])
print("min_df:", best["min_df"])
print("max_df:", best["max_df"])
print("sublinear_tf:", best["sublinear_tf"])
print("F1 Macro:", round(best["f1_macro"], 4))

print("\nReporte de clasificación (mejor experimento):")
print(classification_report(y_test, best["pred"], target_names=class_names, zero_division=0))

Mejor experimento encontrado
Modelo: MultinomialNB
Alpha: 0.5
max_features: 10000
min_df: 3
max_df: 0.9
sublinear_tf: True
F1 Macro: 0.8621

Reporte de clasificación (mejor experimento):
              precision    recall  f1-score   support

       World       0.88      0.87      0.88       122
      Sports       0.95      0.96      0.96       145
    Business       0.78      0.75      0.76       106
    Sci/Tech       0.83      0.87      0.85       127

    accuracy                           0.87       500
   macro avg       0.86      0.86      0.86       500
weighted avg       0.87      0.87      0.87       500



#### Conclusión Item 3

El mejor rendimiento se obtuvo con **MultinomialNB** (`alpha = 0.5`) y una vectorización TF-IDF con `max_features = 10000`, `min_df = 3`, `max_df = 0.9` y `sublinear_tf = True`, logrando un **F1 Macro = 0.8621**.

En términos generales, el modelo clasifica muy bien el conjunto de test (**accuracy = 0.87**). La clase con mejor desempeño es **Sports** (F1 = 0.96), mientras que la más difícil es **Business** (F1 = 0.76), lo que sugiere mayor solapamiento de vocabulario entre noticias de negocios y otras categorías.

Como conclusión, la combinación de filtrado de términos poco frecuentes (`min_df`) y reescalado de frecuencias (`sublinear_tf`) ayudó a mejorar el desempeño global. Además, se cumplió la consigna de **no modificar `ngram_range`**.